In [ ]:
import pandas as pd
import joblib

from self_healing.health_monitor import HealthMonitor
from self_healing.drift_detector import DriftDetector
from self_healing.decision_engine import DecisionEngine
from self_healing.retrainer import Retrainer
from self_healing.rollback import RollbackManager
from attacks.attack_controller import AttackController


In [ ]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test  = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test  = pd.read_csv("../data/processed/y_test.csv").squeeze()

y_train_bin = (y_train != "benign").astype(int)
y_test_bin  = (y_test  != "benign").astype(int)

model = joblib.load("../models/baseline/random_forest_v1.joblib")


In [ ]:
attacker = AttackController({"drift_strength": 0.4})
X_attacked = attacker.apply_drift(X_test)

y_pred = model.predict(X_attacked)


In [ ]:
health_monitor = HealthMonitor(recall_threshold=0.85)
drift_detector = DriftDetector()

health = health_monitor.evaluate(y_test_bin, y_pred)
drift  = drift_detector.detect(X_test, X_attacked)

health, drift


In [ ]:
decision_engine = DecisionEngine()
decision = decision_engine.decide(health, drift)

decision


In [ ]:
if decision == "retrain":
    retrainer = Retrainer()
    new_model = retrainer.retrain(X_train, y_train_bin)
    retrainer.save(new_model, "../models/baseline/random_forest_healed.joblib")
    print("Model retrained and healed")

elif decision == "rollback":
    rollback = RollbackManager()
    model = rollback.rollback("../models/baseline/random_forest_v1.joblib")
    print("Rolled back to safe checkpoint")

else:
    print("System healthy — no action taken")
